# 05 · Validation Plan — per-subunit SPR + cell STAT-phosphorylation + controls

**Standard slot:** *validation plan.* **For Project 13 this means:** turn the selective candidates into
a **costed, controlled wet-lab plan** — **per-subunit SPR/BLI** (K_D to IL-2Rα, IL-2Rβ, γc *separately*
→ confirm selectivity), **a cell-based STAT-phosphorylation (pSTAT5) assay** (the readout that decides
agonism, because **binding ≠ signaling**), the mandatory controls (positive: native IL-2 / Neo-2/15;
**scrambled-interface** negative; unrelated negative), an expression strategy, a **thermostability**
comparison to native IL-2, and the **Boltz-2 affinity** stretch (scaffold only) (D4/D5).

A design that passes every filter and looks selective is a **hypothesis** — per-subunit SPR + pSTAT5 is
what tests it. Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the experimental validation plan

Generate a plan card from the top selective candidates: per-subunit assays, the signaling readout,
controls, expression, timeline, costed reagents. Fill the `<...>` from your own numbers; this is the
deliverable other people will actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if (n_top and "paradigm" in top.columns) else {}

plan = f"""# IL-2 Cytokine-Mimetic Validation Plan (Project 13 — by <your name>, <date>)

## Candidates
Top {n_top} SELECTIVE-AGONIST candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, not affinity;
SELECTIVE in silico is not an agonist — BINDING IS NOT SIGNALING. Do NOT fabricate an EC50/K_D.

## Expression strategy
- Mimetic: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa), de novo ->
  high yield + high stability expected (the Neo-2/15 advantage).
- Receptor-subunit ectodomain reagents (IL-2Ra/CD25, IL-2Rb/CD122, gammaC/CD132): mammalian/insect
  expression or commercial; confirm each is active before testing.

## Assays (go/no-go -> selectivity -> signaling)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse?).
2. SELECTIVITY (per-subunit SPR/BLI): measure K_D + kinetics to IL-2Ra, IL-2Rb, AND gammaC SEPARATELY.
   Expect: binds IL-2Rb and gammaC; does NOT (or weakly) bind IL-2Ra -> confirms the βγ-bias in vitro.
3. SIGNALING (the decisive readout): cell-based STAT-phosphorylation (pSTAT5) assay on IL-2-responsive
   cells (e.g., CTLL-2 or primary T/NK). Dose-response -> EC50, Emax (full vs partial agonist?).
   Run on CD25+ vs CD25- cells to confirm alpha-INDEPENDENT signaling (the selectivity payoff:
   activates effector cells, spares CD25-high Tregs).
4. Stability: DSF (Tm) vs native IL-2 -> quantify the thermostability advantage.
   Deep (optional): co-crystal / cryo-EM of the mimetic-receptor complex; in-vivo Treg-vs-effector.

## Controls (MANDATORY)
- Positive: native IL-2 (and/or Neo-2/15) -> confirms SPR reagents + the pSTAT5 assay/cells respond.
- Negative (scrambled-interface): YOUR OWN top design with its beta/gammaC interface residues
  scrambled/mutated -> must LOSE binding AND signaling (cleanest specificity control).
- Negative (unrelated): an unrelated mini-protein of similar size -> should not bind or signal.

## Realistic expectations
De novo agonist design with clean subunit selectivity is HARD. In-silico hit rates vary widely; the
MAJORITY of in-silico hits fail experimentally, and even an experimental BINDER may fail to SIGNAL
(wrong dimerizing geometry). Report the experimental hit rate honestly. A selective binder that does
NOT trigger pSTAT5 is a negative result worth reporting.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} mimetics + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- Receptor-subunit reagents (a/b/gammaC) + SPR/BLI chips + native IL-2 positive control: $<...>.
- pSTAT5 assay (antibodies, IL-2-responsive cells, CD25+/- lines, flow time): $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
A receptor-selective agonist mimicking a human cytokine for cancer immunotherapy / immune modulation,
whose explicit goal is to REDUCE the toxicity of native IL-2 (βγ-biased -> spares CD25-high Tregs and
vascular-leak toxicity) -> in scope, LOW dual-use risk. Gene synthesis via a biosecurity-screening
provider; cell-based immune assays under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its β/γc interface
residues** (the positions contacting the signaling chains) — it should **lose** binding **and**
signaling. Generating these alongside the real designs (same expression batch) makes the SPR + pSTAT5
comparison airtight. Here we scaffold the sequence-level scramble deterministically; on Colab, scramble
the predicted **interface** positions specifically using the predicted contacts.

In [ ]:
import random
import cytokine_tools as ct   # ct._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting β/γc)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s:
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=ct._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control (must lose binding AND signaling)"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

## 3 · (Stretch) Boltz-2 affinity per subunit on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes — run it **per subunit** to
corroborate the selectivity ranking. Use it for **relative ranking + caveats only** — **never fabricate
a K_D/EC50**, and never present a predicted number as measured. This tells you which hits to test
first, not whether they bind or signal.

In [ ]:
# Scaffold ONLY. Do NOT invent affinities. On Colab:
#   pip install boltz; build each (mimetic, subunit) complex input for α/β/γc; run boltz predict with
#   affinity mode; read the predicted-affinity signal PER SUBUNIT and report the RELATIVE ranking of the
#   top hits + heavy caveats (corroborates the AF2 selectivity profile).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking per subunit + caveats only, NEVER a fabricated K_D/EC50.")
print("Use it to PRIORITIZE which selective hits to test first in SPR/pSTAT5 — not as evidence of binding or signaling.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **per-subunit SPR/BLI** + the **cell pSTAT5 signaling** assay, expression, timeline, costed reagents.
- [ ] Controls specified: positive (native IL-2 / Neo-2/15), **scrambled-interface** negative (`results/negative_controls.csv`), unrelated negative.
- [ ] Selectivity confirmed in vitro (SPR to α/β/γc separately) AND agonism tested (pSTAT5, CD25± cells).
- [ ] Thermostability vs native IL-2 planned (DSF Tm); (stretch) Boltz-2 per-subunit ranking only — no fabricated K_D/EC50.
- [ ] Honest framing: every design is a hypothesis; selective in silico ≠ agonist; binding ≠ signaling; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a subunit-selective agonist design + selectivity profile + a controlled per-subunit-SPR +
STAT-signaling validation plan, following the binder-family template with a selectivity twist.